# MM-IMDB Multimodal Experiments

Run each cell in order. Every experiment writes its metrics to `results/<name>.json`.

Equivalent to running each `python main.py <subcommand>` from the command line, but you can tweak hyperparameters per cell.

**Prerequisite:** [uv](https://docs.astral.sh/uv/) installed. Run the **Environment setup** cell below once; it creates the venv, installs deps, downloads the dataset, and registers a `mm-imdb-uv` Jupyter kernel.

## Environment setup (run once)

Uses [uv](https://docs.astral.sh/uv/) to: (1) create `.venv/` and install dependencies from `pyproject.toml` / `uv.lock`, (2) download `tiny-mm-imdb` into `dataset/`, and (3) register the venv as a Jupyter kernel named **Python (mm-imdb uv)**.

Run this cell once with any kernel that has `uv` on `PATH`, then switch the notebook kernel to **Python (mm-imdb uv)** (Kernel → Change kernel) before running the cells below.

If `uv` isn't installed yet: `curl -LsSf https://astral.sh/uv/install.sh | sh`.

In [ ]:
import shutil, subprocess

if shutil.which("uv") is None:
    raise SystemExit(
        "uv not found on PATH. Install it first:\n"
        "  curl -LsSf https://astral.sh/uv/install.sh | sh\n"
        "Then restart the kernel and re-run this cell."
    )

def run(cmd):
    print(">", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

# 1) Install/refresh the venv from pyproject.toml + uv.lock
run(["uv", "sync"])

# 2) Download tiny-mm-imdb (idempotent — script no-ops if dataset already exists)
run(["uv", "run", "python", "-m", "src.data.get"])

# 3) Register the venv as a Jupyter kernel
run([
    "uv", "run", "python", "-m", "ipykernel", "install",
    "--user", "--name", "mm-imdb-uv",
    "--display-name", "Python (mm-imdb uv)",
])

print(
    "\nSetup complete. Switch the notebook kernel to 'Python (mm-imdb uv)' "
    "via Kernel → Change kernel, then run the cells below."
)

## Setup

Auto-reload so edits to `src/` are picked up without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

from types import SimpleNamespace
import torch
import main

# Common hyperparameters; override per cell if needed.
def args(**overrides):
    base = dict(
        epochs=20,
        batch_size=128,
        lr=1e-3,
        device="auto",
        seed=0,
        label_fraction=1.0,
    )
    base.update(overrides)
    return SimpleNamespace(**base)

print("CUDA available:", torch.cuda.is_available())

## Step 0 — Inspect dataset

Prints split sizes and genre distribution from the CSV.

In [ ]:
main.cmd_inspect_data(args())

## Experiment 1 — Text-only baseline

Frozen BERT encoder (768-d [CLS]) → MLP classifier. Isolates the contribution of the text modality.

In [ ]:
main.cmd_text_only(args(epochs=20))

## Experiment 2 — Image-only baseline

Frozen ResNet50 encoder (2048-d pooled) → MLP classifier. Expected to underperform text-only on Macro-F1.

In [ ]:
main.cmd_image_only(args(epochs=20))

## Experiment 3 — Early fusion (concat + MLP)

Concatenate BERT and ResNet50 embeddings (768+2048=2816-d), pass through an MLP. Cross-modal interactions are learned from the outset.

In [ ]:
main.cmd_fusion_early(args(epochs=20))

## Experiment 4 — Late fusion (probability averaging)

Independent per-modality classifiers; average the 23-dimensional logits. Robust to missing modalities at inference.

In [ ]:
main.cmd_fusion_late(args(epochs=20))

## Experiment 5 — Gated Multimodal Unit (GMU)

Input-dependent gates choose per-unit how much each modality contributes. Arevalo et al.'s reported baseline: Micro-F1 = 0.630, Macro-F1 = 0.541.

In [ ]:
main.cmd_fusion_gmu(args(epochs=20))

## Experiment 6 — Low-label supervised baseline (GMU @ 20%)

Same GMU model as experiment 5 but trained on only 20% of the labels. Sets the floor for the SSL experiments below.

In [ ]:
main.cmd_semi_baseline(args(
    epochs=20,
    label_fraction=0.2,
    pseudo_threshold=0.9,
    consistency_weight=1.0,
    ema_alpha=0.999,
))

## Experiment 7 — Semi-supervised: pseudo-labeling

Adds a confidence-thresholded pseudo-label loss on the 80% unlabeled samples. Only accepts samples that are confident on all 23 classes.

In [ ]:
main.cmd_semi_pseudo(args(
    epochs=20,
    label_fraction=0.2,
    pseudo_threshold=0.9,
    consistency_weight=1.0,
    ema_alpha=0.999,
))

## Experiment 8 — Semi-supervised: Mean Teacher

EMA teacher network produces a consistency target on perturbed feature views. Softer signal than hard pseudo-labels.

In [ ]:
main.cmd_semi_meanteacher(args(
    epochs=20,
    label_fraction=0.2,
    pseudo_threshold=0.9,
    consistency_weight=1.0,
    ema_alpha=0.999,
))

## Experiment 9 — Self-supervised: cross-modal InfoNCE + linear probe

Pretrain text and image projection heads with symmetric InfoNCE on all data (no labels). Then freeze and train a linear probe on 20% of labels.

In [ ]:
main.cmd_selfsup_contrastive(args(
    epochs=20,
    label_fraction=0.2,
    temperature=0.07,
))

## Summary — Collect all results into one table

Reads every `results/*.json` produced above and prints the test F1 scores side-by-side. Copy these numbers into `docs/chapters/results.typ` where the `[TODO]` markers are.

In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []
for p in sorted(Path("results").glob("*.json")):
    payload = json.loads(p.read_text())
    f1 = payload["test"]["f1"]
    rows.append({
        "experiment": payload["experiment"],
        "micro": f1["micro"],
        "macro": f1["macro"],
        "weighted": f1["weighted"],
        "samples": f1["samples"],
    })

df = pd.DataFrame(rows)
df = df.round(4)
print(df.to_string(index=False))

## Optional — Learning curves

Plot train/dev loss and dev Macro-F1 over epochs for each fully-supervised experiment.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

supervised_runs = [
    "text-only", "image-only", "fusion-early", "fusion-late", "fusion-gmu",
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in supervised_runs:
    p = Path("results") / f"{name}.json"
    if not p.exists():
        continue
    payload = json.loads(p.read_text())
    history = payload["history"]
    epochs = range(1, len(history["train_loss"]) + 1)
    axes[0].plot(epochs, history["train_loss"], label=f"{name} train")
    axes[0].plot(epochs, history["dev_loss"], linestyle="--", label=f"{name} dev")
    macro_f1 = [f["macro"] for f in history["dev_f1"]]
    axes[1].plot(epochs, macro_f1, label=name)

axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(fontsize=8); axes[0].set_title("Loss")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("dev Macro-F1"); axes[1].legend(fontsize=8); axes[1].set_title("Dev Macro-F1")
plt.tight_layout()
plt.show()